In [1]:
#https://pygithub.readthedocs.io/en/stable/introduction.html
from github import Github
# Authentication is defined via github.Auth
from github import Auth
import pandas as pd
import numpy as np
import json 
from datetime import datetime, date
from collections import Counter
import plotly.express as px
import time
import pickle
intrinsic_people = ["@aaronchongth","@akash-roboticist","@andreasBihlmaier","@arjo129","@audrow","@azeey","@damon-oss","@faximan","@jennuine","@koonpeng","@kscottz","@luca-della-vedova","@marcoag","@mbordignon-intrinsic","@methylDragon","@mjcarroll","@mjeronimo","@mxgrey","@nuclearsandwich-ai","@quarkytale","@scpeters","@sloretz","@tfoote","@udaya2899","@xiyuoh","@Yadunund"]
org_name = "open-rmf"
this_year = 2024
last_year = 2023

In [2]:
# Grab the access token
with open('./tokens.json',"r") as json_data:
    tokens = json.loads(json_data.read())
    json_data.close()

auth = Auth.Token(tokens["Github"])

# Public Web Github
gh = Github(auth=auth)

In [3]:
org = gh.get_organization(org_name)
repos = org.get_repos()

In [4]:
def extract_contributions(gh, repo_name, start_date, end_date):
# Get github repo level stats for two date ranges
    repo = gh.get_repo(repo_name)
    prs = repo.get_pulls(state='closed', sort='created')
    year = []
    results = {}
    count = 0
    for pr in prs:
        if start_date < pr.closed_at.date() < end_date:
            year.append(pr)
            count += 1
            
    results["repo"] = repo_name
    results["prs"] = year
    results["start_date"] = start_date    
    results["end_date"] = end_date    
    results["users"] = []
    results["handles"] = []
    results["add"] = 0
    results["del"] = 0 
    results["comments"] = 0
    results["files"] = 0
    
    for pr in year:
        results["users"].append(pr.user.name)
        results["handles"].append(pr.user.login)
        results["files"] += pr.changed_files 
        results["del"] += pr.deletions
        results["add"] += pr.additions
        results["comments"] += pr.review_comments

    results["total_prs"] = count
    results["total_users"] = len(set(results["users"]))

    return results

In [5]:
# Create a list of github repos for an org
full_repo_list = []
i = 0
has_repos = True
while has_repos:
    next_repos = repos.get_page(i)
    if len(next_repos) > 0:
        i += 1
        full_repo_list += next_repos
    else:
        has_repos = False
        
print(full_repo_list)
print(len(full_repo_list))

[Repository(full_name="open-rmf/rmf_traffic_editor"), Repository(full_name="open-rmf/free_fleet"), Repository(full_name="open-rmf/rmf-web"), Repository(full_name="open-rmf/traffic_editor_assets"), Repository(full_name="open-rmf/menge_vendor"), Repository(full_name="open-rmf/fleet_adapter_mir"), Repository(full_name="open-rmf/free_fleet_ros2"), Repository(full_name="open-rmf/free_fleet_ros1"), Repository(full_name="open-rmf/rmf-cloud-tools"), Repository(full_name="open-rmf/rmf_simulation"), Repository(full_name="open-rmf/rmf"), Repository(full_name="open-rmf/.github"), Repository(full_name="open-rmf/rmf_demos"), Repository(full_name="open-rmf/rmf_internal_msgs"), Repository(full_name="open-rmf/rmf_battery"), Repository(full_name="open-rmf/rmf_task"), Repository(full_name="open-rmf/rmf_traffic"), Repository(full_name="open-rmf/rmf_ros2"), Repository(full_name="open-rmf/rmf_utils"), Repository(full_name="open-rmf/rmf_cmake_uncrustify"), Repository(full_name="open-rmf/ament_cmake_catch2"),

In [6]:
this_year_start = date(this_year, 1, 1)
this_year_end = date(this_year, 12, 31)
last_year_start = date(last_year, 1, 1)
last_year_end = date(last_year, 12, 31)
full_org_results = {}
fname = 'github_stats_{0}_{1}-{2}.pkl'.format(org_name,this_year,last_year)
for repo in full_repo_list:
    print("Extracting data for {0} from {1} to {2}".format(repo.full_name,this_year_start,this_year_end))
    this_year_results = extract_contributions(gh, repo.full_name, this_year_start, this_year_end)
    print("Extracting data for {0} from {1} to {2}".format(repo.full_name,last_year_start,last_year_end))
    last_year_results = extract_contributions(gh, repo.full_name, last_year_start, last_year_end)
    full_org_results[repo.name] = {}
    full_org_results[repo.name][this_year] = this_year_results
    full_org_results[repo.name][last_year] = last_year_results
    with open(fname,"wb") as file:
        pickle.dump(full_org_results, file)
        print("Wrote: {0}".format(fname))
    print("-----------------------------")



Extracting data for open-rmf/rmf_traffic_editor from 2024-01-01 to 2024-12-31
Extracting data for open-rmf/rmf_traffic_editor from 2023-01-01 to 2023-12-31
Wrote: github_stats_Organization(login="open-rmf")_2024-2023.pkl
-----------------------------
Extracting data for open-rmf/free_fleet from 2024-01-01 to 2024-12-31
Extracting data for open-rmf/free_fleet from 2023-01-01 to 2023-12-31
Wrote: github_stats_Organization(login="open-rmf")_2024-2023.pkl
-----------------------------
Extracting data for open-rmf/rmf-web from 2024-01-01 to 2024-12-31
Extracting data for open-rmf/rmf-web from 2023-01-01 to 2023-12-31
Wrote: github_stats_Organization(login="open-rmf")_2024-2023.pkl
-----------------------------
Extracting data for open-rmf/traffic_editor_assets from 2024-01-01 to 2024-12-31
Extracting data for open-rmf/traffic_editor_assets from 2023-01-01 to 2023-12-31
Wrote: github_stats_Organization(login="open-rmf")_2024-2023.pkl
-----------------------------
Extracting data for open-rmf

Request GET /repos/open-rmf/bevy_impulse/pulls/11 failed with 403: Forbidden
Setting next backoff to 2096.70146s


Extracting data for open-rmf/bevy_impulse from 2023-01-01 to 2023-12-31
Wrote: github_stats_Organization(login="open-rmf")_2024-2023.pkl
-----------------------------
Extracting data for open-rmf/smart_cart_api_server from 2024-01-01 to 2024-12-31
Extracting data for open-rmf/smart_cart_api_server from 2023-01-01 to 2023-12-31
Wrote: github_stats_Organization(login="open-rmf")_2024-2023.pkl
-----------------------------
Extracting data for open-rmf/rmf_workcell from 2024-01-01 to 2024-12-31
Extracting data for open-rmf/rmf_workcell from 2023-01-01 to 2023-12-31
Wrote: github_stats_Organization(login="open-rmf")_2024-2023.pkl
-----------------------------
Wrote: github_stats_2024.pkl
-----------------------------
Extracting data for ros2/domain_bridge from 2024-01-01 to 2024-12-31
Extracting data for ros2/domain_bridge from 2023-01-01 to 2023-12-31
Wrote: github_stats_2024.pkl
-----------------------------
Extracting data for ros2/ros_network_viz from 2024-01-01 to 2024-12-31
Extracting

In [7]:
out =  None
with open(fname, 'rb') as file:
        out = pickle.load(file)      
print(len(out.keys()))
print(out.keys())

76
dict_keys(['rmf_traffic_editor', 'free_fleet', 'rmf-web', 'traffic_editor_assets', 'menge_vendor', 'fleet_adapter_mir', 'free_fleet_ros2', 'free_fleet_ros1', 'rmf-cloud-tools', 'rmf_simulation', 'rmf', '.github', 'rmf_demos', 'rmf_internal_msgs', 'rmf_battery', 'rmf_task', 'rmf_traffic', 'rmf_ros2', 'rmf_utils', 'rmf_cmake_uncrustify', 'ament_cmake_catch2', 'traffic-editor-js', 'rmf_docs', 'rmf_visualization_msgs', 'rmf_visualization', 'ts_ros', 'rmf_building_map_msgs', 'rmf_freespace_planner', 'mock-lift-io', 'stubborn_buddies', 'ros2-bridge', 'rmf_gym', 'magni_nav_ros1', 'fleet_adapter_template', 'door_adapter_template', 'rmf_fullstack_installer', 'rmf-gateway-app', 'rmf_massrobotics', 'webrtc-robot-gateway', 'lift_adapter_template', 'rmf-arkit-ios', 'rmf-gateway-pi', 'rmf-panel-js', 'rmf_geometry_testbed', 'rmf_api_msgs', 'nlohmann_json_schema_validator_vendor', 'traffic_editor_iii', 'rmf_integration_tests', 'TemiFleetAdapterBridge', 'temi_fleet_adapter_python', 'rmf_rtls', 'pybi

In [8]:
# Do full org aggregation
to_agg = ["users","add","del","files"]

full_results = {}
full_results[this_year] = {}
full_results[last_year] = {}

first = True
for key in full_org_results.keys():
    if first:
        full_results[this_year] = {k: full_org_results[key][this_year][k] for k in to_agg}
        full_results[last_year] = {k: full_org_results[key][last_year][k] for k in to_agg}
        full_results[this_year]["prs"] = len(full_org_results[key][this_year]["prs"])
        full_results[last_year]["prs"] = len(full_org_results[key][last_year]["prs"])
        first = False
    else:        
        for a in to_agg:
            full_results[this_year][a] += full_org_results[key][this_year][a]
            full_results[last_year][a] += full_org_results[key][last_year][a]
            full_results[this_year]["prs"] += len(full_org_results[key][this_year]["prs"])
            full_results[last_year]["prs"] += len(full_org_results[key][last_year]["prs"])

full_results[last_year]["contributors"] = set(full_results[last_year]["users"])
full_results[last_year]["users"] = len(set(full_results[last_year]["users"]))

full_results[this_year]["contributors"] = set(full_results[this_year]["users"])
full_results[this_year]["users"] = len(set(full_results[this_year]["users"]))


print("Results for {0} ==> {1}".format(last_year,this_year))
print("-------------------------")

temp_this = {}
temp_last = {}
temp_change = {}

for k in full_results[this_year].keys():
    if k == "contributors":
        continue
    change = -100*(full_results[last_year][k]-full_results[this_year][k])/full_results[last_year][k]
    temp_this[k] =  full_results[this_year][k]
    temp_last[k] =  full_results[last_year][k]
    temp_change[k] = change
    print("{0:6s}| {1} : {2:<6} | {3} : {4:<6} | {5:4.2f}%".format(k,last_year,full_results[last_year][k],this_year,full_results[this_year][k],change))
print(full_results[this_year]["contributors"])

summary_results = pd.DataFrame(data=[temp_this,temp_last,temp_change])
summary_results.to_csv("{0}-{1}-{2}-GithubContribsSummary.csv".format(org_name,last_year,this_year))

Results for 2023 ==> 2024
-------------------------
users | 2023 : 26     | 2024 : 32     | 23.08%
add   | 2023 : 390795 | 2024 : 334935 | -14.29%
del   | 2023 : 144257 | 2024 : 193057 | 33.83%
files | 2023 : 4961   | 2024 : 5925   | 19.43%
prs   | 2023 : 1966   | 2024 : 2101   | 6.87%
{'Grey', 'Jose Luis Rivero', 'Reuben Thomas', 'Luca Della Vedova', 'Xiyu', 'Matthew Festo', None, 'Akash Vibhute', 'Melih Korkmaz', 'Gary Bey', 'methylDragon', 'Chen Bainian', 'youliang', 'Francesco Fallica', 'Thomas Ung', 'John TGZ', 'Winston H.', 'Guru', 'Teo Koon Peng', 'yadunund', 'Addisu Z. Taddese', 'Kevin Ma', 'Dev Manek', 'Cheng-Wei Chen', 'Arjo Chakravarty', 'Esteban Martinena Guerrero', 'Aaron Chong', 'Avisheet Srivastava', 'Alejandro Hernández Cordero', 'Jun', 'Deleted user', 'GiBeom Ryu'}


In [9]:
target = "add"
agg_result = []
for key in full_org_results.keys():
    a = full_org_results[key][this_year][target]
    b = full_org_results[key][last_year][target]
    delta = 0.00
    if b > 0:
        delta = (-100.0*(b-a)/b)
    temp = {}
    temp["name"] = key
    temp[this_year] = a
    temp[last_year] = b
    temp["change"] = delta
    agg_result.append(temp)
    
newlist = sorted(agg_result, key=lambda d: d[this_year])
newlist.reverse()
print("Results for '{0}' across {1} org".format(target,org))
print("-------------------------------------------------------------------")
for i in newlist:  
    print("{0:24s}| 2023: {1:<8} | 2024: {2:<8} | delta: {3:4.2f}%".format(i["name"][:24],
                                                                           i[last_year],
                                                                           i[this_year],
                                                                           i["change"]))
    
df = pd.DataFrame(data=newlist)
df.to_csv("{0}-{1}-{2}-NewLines.csv".format(org_name,last_year,this_year))

Results for 'add' across Organization(login="open-rmf") org
-------------------------------------------------------------------
rmf-web                 | 2023: 72367    | 2024: 116271   | delta: 60.67%
rmf_deployment_template | 2023: 61       | 2024: 95512    | delta: 156477.05%
rmf_site                | 2023: 46613    | 2024: 39026    | delta: -16.28%
sdf_rust_experimental   | 2023: 1036     | 2024: 15435    | delta: 1389.86%
free_fleet              | 2023: 66       | 2024: 12181    | delta: 18356.06%
bevy_impulse            | 2023: 0        | 2024: 10926    | delta: 0.00%
rmf_ros2                | 2023: 24775    | 2024: 9907     | delta: -60.01%
rmf_reservation         | 2023: 0        | 2024: 8246     | delta: 0.00%
rmf_demos               | 2023: 7675     | 2024: 5977     | delta: -22.12%
rmf_simulation          | 2023: 903      | 2024: 4727     | delta: 423.48%
fleet_adapter_mir       | 2023: 201340   | 2024: 4316     | delta: -97.86%
smart_cart_api_server   | 2023: 0        | 202

In [10]:
target = "prs"

agg_result = []
for key in full_org_results.keys():
    a = len(full_org_results[key][2024][target])
    b = len(full_org_results[key][2023][target])
    delta = 0.00
    if b > 0:
        delta = (-100.0*(b-a)/b)
    temp = {}
    temp["name"] = key
    temp[this_year] = a
    temp[last_year] = b
    temp["change"] = delta
    agg_result.append(temp)
    
newlist = sorted(agg_result, key=lambda d: d[this_year])
newlist.reverse()
print("PR count by year")
print("Results for '{0}' across ROS 2 org".format(target))
print("-------------------------------------------------------------------")
for i in newlist:  
    print("{0:24s}| 2023: {1:<8} | 2024: {2:<8} | delta: {3:4.2f}%".format(i["name"][:24],
                                                                           i[last_year],
                                                                           i[this_year],
                                                                           i["change"]))
df = pd.DataFrame(data=newlist)
df.to_csv("{0}-{1}-{2}-PRS.csv".format(org_name,last_year,this_year))

PR count by year
Results for 'prs' across ROS 2 org
-------------------------------------------------------------------
rmf-web                 | 2023: 186      | 2024: 141      | delta: -24.19%
rmf_ros2                | 2023: 43       | 2024: 73       | delta: 69.77%
rmf_demos               | 2023: 19       | 2024: 54       | delta: 184.21%
rmf_site                | 2023: 63       | 2024: 32       | delta: -49.21%
rmf_traffic_editor      | 2023: 14       | 2024: 29       | delta: 107.14%
smart_cart_api_server   | 2023: 0        | 2024: 24       | delta: 0.00%
rmf_simulation          | 2023: 14       | 2024: 23       | delta: 64.29%
rmf_internal_msgs       | 2023: 7        | 2024: 18       | delta: 157.14%
bevy_impulse            | 2023: 0        | 2024: 17       | delta: 0.00%
rmf_task                | 2023: 32       | 2024: 14       | delta: -56.25%
rmf_visualization       | 2023: 13       | 2024: 13       | delta: -0.00%
rmf                     | 2023: 12       | 2024: 11       | de